# Check the URLs of the master catalog

It reads the `url` column of `master_catalog.xlsx` and checks that each page answers 200.

In [1]:
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import requests

CATALOG = "output/master/master_catalog.xlsx"
OUT = "output/master/url_check.csv"
WORKERS = 8  # keep it low, so the site does not block us

## Read the URLs

In [2]:
catalog = pd.read_excel(CATALOG, sheet_name="catalog")
urls = catalog["url"].dropna().drop_duplicates().tolist()
print(len(urls), "different URLs")

10726 different URLs


## Check them

Request URL
https://slotcatalog.com/en/slots/gates-of-olympus-super-scatter
Request method
GET
Status code
200 OK
Remote address
104.26.1.173:443
Referrer policy
strict-origin-when-cross-origin
alt-svc
h3=":443"; ma=86400
cache-control
public, max-age=10800
cf-cache-status
DYNAMIC
cf-ray
a55b761bae480b7a-MAD
content-encoding
br
content-type
text/html; charset=UTF-8
date
Thu, 05 Nov 2026 11:19:04 GMT
expires
Thu, 05 Nov 2026 14:19:04 GMT
last-modified
Fri, 30 Oct 2026 14:39:35 GMT
nel
{"report_to":"cf-nel","success_fraction":0.0,"max_age":604800}
priority
u=0,i
report-to
{"group":"cf-nel","max_age":604800,"endpoints":[{"url":"https://a.nel.cloudflare.com/report/v4?s=1J7X%2BcDYflZf%2BWSkDVUox5%2FJAuJMonPt49SPamw12AXQuEJIOVb7suZPDgYGfyd6Ukh8bRHLfMizL97mpqKahS3CKme%2BYiN1haEmUcffl2oI1SbuoCno%2F6cijVu7kctmag%3D%3D"}]}
server
cloudflare
server-timing
cfExtPri
strict-transport-security
max-age=31536000; includeSubDomains; preload
vary
accept-encoding
x-content-type-options
nosniff
:authority
slotcatalog.com
:method
GET
:path
/en/slots/gates-of-olympus-super-scatter
:scheme
https
accept
text/html,application/xhtml+xml,application/xml;q=0.9,image/jxl,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7
accept-encoding
gzip, deflate, br, zstd
accept-language
es-ES,es;q=0.9
cache-control
max-age=0
cookie
PHPSESSID=1i5uspbh0kn64a6k91ouv2o5eq; drnkVisitorId=24820b2c-586a-42df-85c8-bff61a2725f0; popupCnt=0; _ga_V8JCB553MD=GS2.1.s1793877499$o1$g0$t1793877499$j60$l0$h0; _ga=GA1.1.2005811219.1793877500; cf_clearance=Cnr3CDVQ_6y6-EqKWJsmgcti9-RAHA6ul2ueWNpCXtWllPDG2K6T1vUPkm0FydCOj637UkNUt7M6tXbmpsHbitYCdsb55UdDEfTJ4y1Gq0GYRoBd28l9uGMem160ydU5ILt_UqFUGFnA9w6gwPA0JP7NYvQO9tA-C6MzgTCNXN-kG7D2xv_kKln9gwiB7MpXWOnnTxVpgK_cezUh3dnrXmYqGJYV04fcFesVxlNcjpZHr4VOTFOVx7VHURStJHBxOIsdwB1bM8SQ1zk3NmtYeNdpyjvcAOP3-i-9xt80vyhZs78; g_state={"i_l":0,"i_ll":1793877499830,"i_b":"7MPVGgJg/39x1KgYPEj1ByoY15QLnDMzfXpiBnD9w3A","i_e":{"enable_itp_optimization":24},"i_et":1793877499830}; isPromoBannerClicked=1
if-modified-since
Fri, 30 Oct 2026 14:39:35 GMT
priority
u=0, i
sec-ch-ua
"Not-A.Brand";v="99", "Google Chrome";v="157", "Chromium";v="157"
sec-ch-ua-mobile
?0
sec-ch-ua-platform
"Windows"
sec-fetch-dest
document
sec-fetch-mode
navigate
sec-fetch-site
none
sec-fetch-user
?1
upgrade-insecure-requests
1
user-agent
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/157.0.0.0 Safari/537.36

In [3]:
session = requests.Session()
session.headers["User-Agent"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/157.0.0.0 Safari/537.36"


def check(url):
    try:
        r = session.get(url, timeout=20, stream=True)  # stream: we do not download the page
        r.close()
        return {"url": url, "status": r.status_code, "final_url": r.url}
    except requests.RequestException as e:
        return {"url": url, "status": None, "final_url": type(e).__name__}


start = time.time()
with ThreadPoolExecutor(WORKERS) as pool:
    results = pd.DataFrame(pool.map(check, urls))
print(f"{time.time() - start:.0f} seconds")

320 seconds


## Result

In [4]:
results.to_csv(OUT, index=False)
results["status"].value_counts(dropna=False)

status
429.0    10417
200.0      308
NaN          1
Name: count, dtype: int64

In [5]:
# URLs that do not return 200
results[results["status"] != 200]

,url,status,final_url
0,https://slotcatalog.com/en/slots/gates-of-olym...,403.0,https://slotcatalog.com/en/slots/gates-of-olym...
1,https://slotcatalog.com/en/slots/Gates-of-Olym...,403.0,https://slotcatalog.com/en/slots/Gates-of-Olym...
2,https://slotcatalog.com/en/slots/sweet-bonanza...,403.0,https://slotcatalog.com/en/slots/sweet-bonanza...
3,https://slotcatalog.com/en/slots/Aviatrix,403.0,https://slotcatalog.com/en/slots/Aviatrix
4,https://slotcatalog.com/en/slots/Sweet-Bonanza,403.0,https://slotcatalog.com/en/slots/Sweet-Bonanza
...,...,...,...
10721,https://slotcatalog.com/en/slots/the-library,403.0,https://slotcatalog.com/en/slots/the-library
10722,https://slotcatalog.com/en/slots/triple-money-...,403.0,https://slotcatalog.com/en/slots/triple-money-...
10723,https://slotcatalog.com/en/slots/triple-money-...,403.0,https://slotcatalog.com/en/slots/triple-money-...
10724,https://slotcatalog.com/en/slots/triple-money-...,403.0,https://slotcatalog.com/en/slots/triple-money-...


In [5]:
# Same list, with the game
games = catalog[["whizdom_game_id", "game_name", "provider", "url"]]
games.merge(results[results["status"] != 200], on="url")

,whizdom_game_id,game_name,provider,url,status,final_url
0,bc817bc2-d89b-5326-a38c-5adf953f50ce,Pink Joker Hold And Win,playson,https://slotcatalog.com/en/slots/pink-joker-ho...,429.0,https://slotcatalog.com/en/slots/pink-joker-ho...
1,b45b5c5e-abce-5fef-a5d4-3981744e2417,Gates Of Olympus Xmas 1000,pragmatic-play,https://slotcatalog.com/en/slots/gates-of-olym...,429.0,https://slotcatalog.com/en/slots/gates-of-olym...
2,892b14dc-5f83-5565-ad7b-e81b20395da3,Mexican Coins Cash Up,fugaso,https://slotcatalog.com/en/slots/mexican-coins,429.0,https://slotcatalog.com/en/slots/mexican-coins
3,255dc183-2571-585b-acde-e92d493485fc,Black Clover Bell Link,egt-digital,https://slotcatalog.com/en/slots/black-clover-...,429.0,https://slotcatalog.com/en/slots/black-clover-...
4,6385b1d9-5707-5a93-bd7c-8f4a65d18d4d,Diamond Cascade,pragmatic-play,https://slotcatalog.com/en/slots/Diamond-Casca...,429.0,https://slotcatalog.com/en/slots/Diamond-Casca...
...,...,...,...,...,...,...
11009,70183ef9-cc97-5195-b5b0-b764ac07152d,Triple Money Jackpot Bells™,playtech-origins,https://slotcatalog.com/en/slots/triple-money-...,429.0,https://slotcatalog.com/en/slots/triple-money-...
11010,a3196172-cac7-522a-85fb-ade089908bf9,Triple Money Jackpot™ Gold!,playtech-origins,https://slotcatalog.com/en/slots/triple-money-...,429.0,https://slotcatalog.com/en/slots/triple-money-...
11011,2f9bc6cd-fb1a-5374-a8f6-f06d1b848d15,Triple Money Jackpot™ Spicy!,playtech-origins,https://slotcatalog.com/en/slots/triple-money-...,429.0,https://slotcatalog.com/en/slots/triple-money-...
11012,bebe11f6-eb31-5b7b-8d6a-b44ecd23d1fd,Wings of Iguazu,pg-soft,https://slotcatalog.com/en/slots/wings-of-iguazu,429.0,https://slotcatalog.com/en/slots/wings-of-iguazu


## Check with the Wayback Machine

Slot Catalog answers 403 to scripts. The Wayback Machine (web.archive.org) saves copies of pages and the HTTP status it got.
For each URL I ask for its copies since 2025 and keep the last status and the last date with 200.
It is slow (about 1 URL per second) and it saves each answer, so you can stop it and run it again.

In [6]:
from pathlib import Path

WAYBACK = Path("output/master/url_wayback.csv")
CDX = "https://web.archive.org/cdx/search/cdx"
SINCE = "2025"

wb_session = requests.Session()
wb_session.headers["User-Agent"] = "whizdom-url-check"


def wayback(url):
    """The copies of one URL in the Wayback Machine: how many, the last status and the last date with 200."""
    params = {"url": url, "output": "json", "fl": "timestamp,statuscode", "from": SINCE}
    for wait in [5, 30, 120]:
        try:
            r = wb_session.get(CDX, params=params, timeout=60)
            if r.status_code == 200:
                rows = r.json()[1:] if r.text.strip() else []  # the first row is the header
                ok = [t for t, s in rows if s == "200"]
                return {"url": url, "captures": len(rows),
                        "last_date": rows[-1][0][:8] if rows else None,
                        "last_status": rows[-1][1] if rows else None,
                        "last_200": ok[-1][:8] if ok else None}
        except requests.RequestException:
            pass
        time.sleep(wait)  # too many requests or no answer: wait and try again
    return {"url": url, "captures": None}

In [12]:
done = set()
if WAYBACK.exists():
    old = pd.read_csv(WAYBACK)
    done = set(old.loc[old["captures"].notna(), "url"])
todo = [u for u in urls if u not in done]
print(len(todo), "URLs to ask")

for i, url in enumerate(todo[:300], 1):
    pd.DataFrame([wayback(url)]).to_csv(WAYBACK, mode="a", header=not WAYBACK.exists(), index=False)
    time.sleep(1)
    if i % 500 == 0:
        print(i, "done")

10724 URLs to ask


## Wayback result

In [17]:
wb = pd.read_csv(WAYBACK, dtype={"last_date": str, "last_status": str, "last_200": str})
wb = wb.drop_duplicates("url", keep="last")


def result(row):
    if pd.isna(row["captures"]):
        return "no answer"
    if row["captures"] == 0:
        return "no copy"
    if pd.notna(row["last_200"]):
        return "200 seen"
    return "never 200"


wb["result"] = wb.apply(result, axis=1)
wb["result"].value_counts()

result
no copy      215
200 seen      72
never 200     14
no answer      1
Name: count, dtype: int64

In [14]:
# Month of the last 200
wb["last_200"].str[:6].value_counts().sort_index()

last_200
202501    1
202503    7
202505    1
202506    3
202507    2
202508    7
202510    4
202511    1
202512    9
202601    2
202602    7
202603    7
202604    9
202606    3
202608    2
202609    7
Name: count, dtype: int64

In [15]:
# Copies but never 200: the status they got (403 is the block, 404 is a page that does not exist)
wb[wb["result"] == "never 200"]["last_status"].value_counts()

last_status
403    14
Name: count, dtype: int64

In [16]:
# URLs with no 200, with the game
games.merge(wb[wb["result"] != "200 seen"], on="url")

,whizdom_game_id,game_name,provider,url,captures,last_date,last_status,last_200,result
0,4b635c9b-6b41-50ec-bcde-c079a1ff2d1a,Gates Of Olympus 1000,pragmatic-play,https://slotcatalog.com/en/slots/Gates-of-Olym...,2.0,20251211,403,NaN,never 200
1,b3c2d29d-3000-5137-87ac-331b968b5f95,Aviatrix,aviatrix,https://slotcatalog.com/en/slots/Aviatrix,0.0,NaN,NaN,NaN,no copy
2,7df07bfb-ab74-5954-b3aa-68d8809b252c,Flaming Hot Extreme Vip Bell Link,egt-digital,https://slotcatalog.com/en/slots/flaming-hot-e...,0.0,NaN,NaN,NaN,no copy
3,fc0f0a98-28bd-5b3b-8059-08eb32f6f075,Immersive Roulette,evolution,https://slotcatalog.com/en/slots/Immersive-Rou...,0.0,NaN,NaN,NaN,no copy
4,06134945-9ee4-5d38-bad6-fd6f37f957c7,Better Barn House Bonanza,pragmatic-play,https://slotcatalog.com/en/slots/better-barnho...,0.0,NaN,NaN,NaN,no copy
...,...,...,...,...,...,...,...,...,...
286,f0b70b9e-87c9-557a-af9d-468d137ea697,20 Super Hot Progressive Jackpot,amusnet,https://slotcatalog.com/en/slots/20-Super-Hot,0.0,NaN,NaN,NaN,no copy
287,cf216277-6efd-5c95-975f-a13e80429672,40 Lucky King,amusnet,https://slotcatalog.com/en/slots/40-Lucky-King,0.0,NaN,NaN,NaN,no copy
288,59c87d8d-3f92-5679-bb30-d7362453cbbc,Dice and roll,amusnet,https://slotcatalog.com/en/slots/Dice-Roll,0.0,NaN,NaN,NaN,no copy
289,f65ab884-4ce8-5046-816c-83d906b5ed49,Treasures of Aztec INF,pg-soft,https://slotcatalog.com/en/slots/Treasures-of-...,0.0,NaN,NaN,NaN,no copy


## Check with the Slot Catalog JSON (November 2025)

The JSON files in `internal_taxonomy/sa_raw` are the pages downloaded in the mapping.
A JSON with a title means the page answered 200 with the game on that day.

In [ ]:
import json

SA_RAW = Path("internal_taxonomy/sa_raw")


def norm(url):
    return str(url).strip().lower().rstrip("/")


pages = []
for f in SA_RAW.glob("*.json"):
    d = json.loads(f.read_text())
    if "source_url" in d:
        ok = d.get("extraction_status") == "success" and bool(d.get("raw_data", {}).get("title"))
        pages.append({"key": norm(d["source_url"]), "json_date": d["extraction_timestamp"][:10], "json_ok": ok})
pages = pd.DataFrame(pages).sort_values("json_ok").drop_duplicates("key", keep="last")

check = pd.DataFrame({"url": urls})
check["key"] = check["url"].map(norm)
check = check.merge(pages, on="key", how="left").drop(columns="key")
check["json_ok"] = check["json_ok"].eq(True)
print(check["json_ok"].mean().round(3), "of the URLs answered 200 in the mapping")
check.groupby("json_ok")["json_date"].agg(["size", "min", "max"])

## Check today with your browser (a sample)

1. Run the next cell: it takes 300 URLs at random.
2. In Chrome open `chrome://net-export` (in Edge `edge://net-export`) and click **Start Logging to Disk**.
3. Run `open_batch(0)`, `open_batch(1)`... It opens 25 tabs each time. Wait until they load and close them.
4. Click **Stop Logging** and save the file in `output/master/netlog/`.

In [ ]:
import subprocess

BATCH = 25
sample = pd.Series(urls).sample(300, random_state=1).tolist()
pd.DataFrame({"url": sample}).to_csv("output/master/url_sample.csv", index=False)


def open_batch(n):
    """Open the batch n (0 to 11) of the sample in your Windows browser."""
    for url in sample[n * BATCH:(n + 1) * BATCH]:
        subprocess.run(["explorer.exe", url])
        time.sleep(2)


print(len(sample) // BATCH, "batches: open_batch(0) ...", f"open_batch({len(sample) // BATCH - 1})")

In [ ]:
open_batch(0)

## Read the log of the browser

In [ ]:
import re

NETLOG = Path("output/master/netlog")

status = {}  # URL -> last HTTP status the browser got
for f in sorted(NETLOG.glob("*.json")):
    log = json.loads(f.read_text(encoding="utf-8"))
    names = {v: k for k, v in log["constants"]["logEventTypes"].items()}
    url_of, first = {}, {}  # request id -> URL it is loading now, and the URL it started with
    for e in log["events"]:
        name, rid, params = names.get(e["type"]), e["source"]["id"], e.get("params", {})
        if name == "URL_REQUEST_START_JOB":
            url_of[rid] = params["url"]
            first.setdefault(rid, params["url"])
        elif name == "HTTP_TRANSACTION_READ_RESPONSE_HEADERS" and rid in url_of:
            code = re.search(r"\b(\d{3})\b", params["headers"][0])
            if code:  # the last one wins: after a redirect or the Cloudflare check
                status[norm(url_of[rid])] = status[norm(first[rid])] = int(code.group(1))
print(len(status), "URLs in the log")

In [ ]:
today = pd.DataFrame({"url": sample})
today["status_today"] = today["url"].map(lambda u: status.get(norm(u)))
today = today.merge(check, on="url", how="left")
today["status_today"].value_counts(dropna=False)

In [ ]:
# Sample URLs with no 200 today, with the game
games.merge(today[today["status_today"] != 200], on="url")